# Feed Forward Network with PyTorch

## AI Usage

I used AI to investigate and understand errors.

## PyTorch setup

In [200]:
import torch

In [201]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device: {device}")

Using device: cuda


## Load data and create Datasets

In [202]:
from torchvision.datasets import MNIST
from torchvision import transforms

In [203]:
transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
train_data

Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor()

In [204]:
from torch.utils.data import Subset

Split the training data into training and validation sets

In [205]:
train_indices = list(range(50000))
val_indices = list(range(50000, len(train_data)))

val_data = Subset(train_data, val_indices)
train_data = Subset(train_data, train_indices)
print(f"Training set size: {len(train_data)}")
print(f"Validation set size: {len(val_data)}")

Training set size: 50000
Validation set size: 10000


In [206]:
test_data = MNIST(root="data", train=False, download=True, transform=transform)
test_data

Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()

## Defining the Neural Network

In [207]:
import torch.nn as nn

In [208]:
input_size = train_data[0][0].numel()

# I think to(device) is not needed here but I added it just in case
model1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model2 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
).to(device)

model3 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(input_size, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 10)
).to(device)

## Defining the optimizer

In [209]:
import torch.optim as optim

In [210]:
optimizer1 = optim.Adam(model1.parameters(), lr=0.001)
optimizer2 = optim.Adam(model2.parameters(), lr=0.01)
optimizer3 = optim.SGD(model3.parameters(), lr=0.001)

## Defining loss function

In [211]:
criterion = nn.CrossEntropyLoss()

## Defining mini-batches creation function

In [212]:
import numpy as np

In [213]:
def create_minibatches(batch_size, x, y):
    total_data = len(x)
    indices = np.arange(total_data)

    for i in range(0, total_data, batch_size):
        batch_idx = indices[i:i + batch_size]
        x_batch = torch.stack([x[j] for j in batch_idx])
        y_batch = torch.tensor([y[j] for j in batch_idx], dtype=torch.long)
        yield x_batch, y_batch

## Train loop

In [214]:
from tqdm import tqdm

In [215]:
def train(model, train_data, optimizer, criterion, epochs, batch_size):
    # Training data
    x_data = [x for x, _ in train_data]
    y_data = [y for _, y in train_data]

    # Validation data
    x_val = [x for x, y in val_data]
    y_val = [y for _, y in val_data]

    for epoch in range(epochs):
        # Train
        model.train()
        running_loss = 0.0

        epoch_iterator = tqdm(
            create_minibatches(batch_size, x_data, y_data),
            total=(len(train_data) + batch_size - 1) // batch_size,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=True
        )

        for inputs, labels in epoch_iterator:
            optimizer.zero_grad()
            # The dataset is using cpu, check if we are using gpu and move to gpu
            if inputs.device != device:
                inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(inputs)

        epoch_train_loss = running_loss / len(train_data)

        # Eval
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, labels in create_minibatches(batch_size, x_val, y_val):
                if inputs.device != device:
                    inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * len(inputs)
        
        epoch_val_loss = val_loss / len(val_data)

        print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_train_loss:.4f}, Val Loss: {epoch_val_loss:.4f}")

    return epoch_train_loss, epoch_val_loss

In [216]:
model1_loss = train(model1, train_data, optimizer1, criterion, epochs=30, batch_size=64)

Epoch 1/30: 100%|██████████| 782/782 [00:03<00:00, 257.84it/s]


Epoch [1/30], Loss: 0.3890, Val Loss: 0.1997


Epoch 2/30: 100%|██████████| 782/782 [00:02<00:00, 265.15it/s]


Epoch [2/30], Loss: 0.1893, Val Loss: 0.1403


Epoch 3/30: 100%|██████████| 782/782 [00:02<00:00, 262.02it/s]


...

Epoch [28/30], Loss: 0.0018, Val Loss: 0.1143


Epoch 29/30: 100%|██████████| 782/782 [00:02<00:00, 261.64it/s]


Epoch [29/30], Loss: 0.0011, Val Loss: 0.1257


Epoch 30/30: 100%|██████████| 782/782 [00:02<00:00, 266.35it/s]


Epoch [30/30], Loss: 0.0019, Val Loss: 0.1528


In [217]:
model2_loss = train(model2, train_data, optimizer2, criterion, epochs=20, batch_size=86)

Epoch 1/20: 100%|██████████| 582/582 [00:02<00:00, 206.51it/s]


Epoch [1/20], Loss: 0.2801, Val Loss: 0.1810


Epoch 2/20: 100%|██████████| 582/582 [00:02<00:00, 209.40it/s]


Epoch [2/20], Loss: 0.1603, Val Loss: 0.1290


Epoch 3/20: 100%|██████████| 582/582 [00:02<00:00, 208.35it/s]


...

Epoch [18/20], Loss: 0.0524, Val Loss: 0.2144


Epoch 19/20: 100%|██████████| 582/582 [00:02<00:00, 212.61it/s]


Epoch [19/20], Loss: 0.0588, Val Loss: 0.1863


Epoch 20/20: 100%|██████████| 582/582 [00:02<00:00, 208.31it/s]


Epoch [20/20], Loss: 0.0499, Val Loss: 0.1956


In [218]:
model3_loss = train(model3, train_data, optimizer3, criterion, epochs=35, batch_size=32)

Epoch 1/35: 100%|██████████| 1563/1563 [00:05<00:00, 290.09it/s]


Epoch [1/35], Loss: 2.2713, Val Loss: 2.2308


Epoch 2/35: 100%|██████████| 1563/1563 [00:05<00:00, 285.68it/s]


Epoch [2/35], Loss: 2.1681, Val Loss: 2.0797


Epoch 3/35: 100%|██████████| 1563/1563 [00:04<00:00, 315.95it/s]


...

Epoch [33/35], Loss: 0.2971, Val Loss: 0.2748


Epoch 34/35: 100%|██████████| 1563/1563 [00:02<00:00, 523.01it/s]


Epoch [34/35], Loss: 0.2935, Val Loss: 0.2717


Epoch 35/35: 100%|██████████| 1563/1563 [00:03<00:00, 482.36it/s]


Epoch [35/35], Loss: 0.2900, Val Loss: 0.2687


In [219]:
for model, (train_loss, val_loss) in zip(
    ["Model 1", "Model 2", "Model 3"],
    [model1_loss, model2_loss, model3_loss]
):
    print(f"{model} - Final Training Loss: {train_loss:.4f}, Final Validation Loss: {val_loss:.4f}")

Model 1 - Final Training Loss: 0.0019, Final Validation Loss: 0.1528
Model 2 - Final Training Loss: 0.0499, Final Validation Loss: 0.1956
Model 3 - Final Training Loss: 0.2900, Final Validation Loss: 0.2687
